# logit-thermo 探索：偽薬（塩）で本当の融点 T_m は下がるか（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 5〜10 分。
- 探索（予想なし）。A2-7b の「なし」「なし（別の種）」「偽薬 2.5%・5%・7.5%」のトークンから、A2-8 と同じ決め方で凍る速さ・溶ける速さを数えて T_m を出し、
  `A2_explore/tm_salt_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：結晶の始まりの探索と A2-8 のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_explore_tm_salt.py`（`60c82f90eb175743…`）は埋め込み。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo 探索 偽薬で本当の融点 T_m は下がるか（A2-7b の下見、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 偽薬で本当の融点 T_m は下がるか（A2-7b の下見、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore Tm salt', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import math
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCON': '0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9', 'SRC8': '06a93c4eec9f3e770ca9e2f3f791a94b37153c908becd0f83d825acfbd11bacd'})
SRCON, SRC8 = _s['SRCON'], _s['SRC8']
SRCTS = '"""探索：偽薬（塩）で本当の融点 T_m は下がるか（A2-7b のトークンの下見、2026-10-30 13:4x）。予想なし。CPU。生成も読み直しもしない。\nscripts/a2_explore_onset.py（copy_marks）と scripts/a2_8_vh.py（phases・doc_counts・fit_vh・A8_TU）を実行したのと同じ名前空間で実行する。\n\nオーナー（2026-10-30 13:37）：（オーナーの承認・発言。原文は非公開の記録にある）\n（13:38）（オーナーの承認・発言。原文は非公開の記録にある）\nやること：A2-7b の「なし」「なし（別の種）」「偽薬 2.5%・5%・7.5%」の腕（温度 0.4〜1.0、サンプル 2、記事 40）で、A2-8 と同じ決め方\n（写しの印・中心の窓・入口 0.5／出口 0.25・凍る回と溶ける回がそれぞれ 10 回以上の温度）で T_m を出し、「なし」との比を記事の組の\n選び直し（同じ記事の選び方を全部の腕で使う）で比べる。写しの禁止の腕は、結晶を見つける印そのものを禁止しているので使わない。\n"""\nimport numpy as np\n\nTS_ARMS = [\'none\', \'none_b\', \'pl025\', \'pl050\', \'pl075\']\nTS_DOSE = {\'none\': 0.0, \'none_b\': 0.0, \'pl025\': 0.025, \'pl050\': 0.05, \'pl075\': 0.075}\nTS_TEMPS = [0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 1.0]\nTS_MODELS = [\'pythia-1b\', \'olmo2-1b\', \'qwen2.5-1.5b\']\nTS_BOOT, TS_SEED = 1000, 20261115\n\n\ndef counts_arm(toks):\n    """toks：(8, S, nD, L) → (17, nD, 10)。U と HT は使わないので 0 を入れる（熱量計の ΔH は 0 になり、n は出ない）。"""\n    nT, S, nD, L = toks.shape\n    out = np.zeros((len(A8_TU), nD, 10))\n    z = np.zeros((S, L))\n    for k, T in enumerate(TS_TEMPS):\n        ku = A8_TU.index(T)\n        for j in range(nD):\n            C = np.stack([copy_marks(toks[k, s, j]) for s in range(S)])\n            out[ku, j] += doc_counts(C, z, z)\n    return out\n\n\ndef tm_salt(base):\n    res = {\'models\': {}, \'arms\': TS_ARMS, \'temps\': TS_TEMPS}\n    for m in TS_MODELS:\n        cnt = {}\n        for a in TS_ARMS:\n            Z = np.load(f\'{base}/A2_7B/{m}@{a}.npz\')\n            assert [float(t) for t in Z[\'temps\']] == TS_TEMPS\n            cnt[a] = counts_arm(Z[\'toks\'])\n        pt = {a: fit_vh(cnt[a].sum(1)) for a in TS_ARMS}\n        use = pt[\'none\'][\'use_idx\']\n        row = {\'point\': {a: dict(Tm=pt[a].get(\'Tm\'), dH_vH=pt[a].get(\'dH_vH\'), use_idx=pt[a][\'use_idx\'], lnK=pt[a][\'lnK\'],\n                                 n_freeze=cnt[a].sum(1)[:, 1].tolist(), n_melt=cnt[a].sum(1)[:, 3].tolist()) for a in TS_ARMS}}\n        # 同じ温度の組（「なし」で使えた温度）で当てはめ直したもの（腕どうしを同じ物差しで比べる）\n        same = {a: fit_vh(cnt[a].sum(1), temps_use=use) for a in TS_ARMS}\n        row[\'same_temps\'] = {a: dict(Tm=same[a].get(\'Tm\'), dH_vH=same[a].get(\'dH_vH\'), ok=same[a][\'ok\']) for a in TS_ARMS}\n        nD = cnt[\'none\'].shape[1]\n        rng = np.random.default_rng(TS_SEED)\n        y = {a: [] for a in TS_ARMS if a != \'none\'}\n        fail = 0\n        for _ in range(TS_BOOT):\n            w = np.bincount(rng.integers(0, nD, nD), minlength=nD).astype(float)\n            f = {a: fit_vh(np.tensordot(cnt[a], w, axes=([1], [0])), temps_use=use) for a in TS_ARMS}\n            if not all(f[a][\'ok\'] and np.isfinite(f[a][\'Tm\']) for a in TS_ARMS):\n                fail += 1\n                continue\n            for a in y:\n                y[a].append(float(np.log(f[a][\'Tm\'] / f[\'none\'][\'Tm\'])))\n        row[\'y_logTm\'] = {a: dict(point=(float(np.log(same[a][\'Tm\'] / same[\'none\'][\'Tm\'])) if same[a][\'ok\'] and same[\'none\'][\'ok\'] else None),\n                                  ci=[float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] if len(v) >= 20 else None)\n                          for a, v in y.items()}\n        row[\'boot_fail\'] = fail / TS_BOOT\n        res[\'models\'][m] = row\n    return res\n\n\ndef self_test_tm_salt():\n    """作り物：凍る速さを濃さで下げた鎖 → 偽薬の腕の T_m が「なし」より下がることを拾う。"""\n    import math\n    rng = np.random.default_rng(0)\n    L, nD, S = 512, 40, 2\n\n    def make(shift):\n        toks = np.zeros((len(TS_TEMPS), S, nD, L), np.int64)\n        for k, T in enumerate(TS_TEMPS):\n            lnK = 6.0 / T - 6.0 / (0.6 - shift)\n            km = 0.01 * math.exp(-lnK / 2); kf = 0.01 * math.exp(lnK / 2)\n            for s in range(S):\n                for j in range(nD):\n                    st, seq, per = 0, [], list(rng.integers(100, 30000, 7))\n                    for t in range(L):\n                        if t > 0:\n                            st = (1 if rng.random() < kf else 0) if st == 0 else (0 if rng.random() < km else 1)\n                        seq.append(per[t % 7] if st else int(rng.integers(100, 30000)))\n                    toks[k, s, j] = seq\n        return toks\n    a, b = counts_arm(make(0.0)), counts_arm(make(0.06))\n    ta, tb = fit_vh(a.sum(1))[\'Tm\'], fit_vh(b.sum(1))[\'Tm\']\n    assert 0.55 < ta < 0.66 and 0.49 < tb < 0.60 and tb < ta - 0.03, (ta, tb)\n    return \'OK\'\n'
assert hashlib.sha256(SRCTS.encode()).hexdigest() == '60c82f90eb17574306e78ca508b3390fa3f9b69c8bf4041b4dff989013ef8884'
exec(SRCON)
exec(SRC8)
exec(SRCTS)
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): o = o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('自己テスト：', self_test_onset(), self_test_tm_salt())


In [ ]:
# セル3：数えて保存（結果は表示しない）
try:
    r = tm_salt('/content/drive/MyDrive/logit-thermo')
    r.update(onset_sha256='0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9', a2_8_sha256='06a93c4eec9f3e770ca9e2f3f791a94b37153c908becd0f83d825acfbd11bacd', tm_salt_sha256='60c82f90eb17574306e78ca508b3390fa3f9b69c8bf4041b4dff989013ef8884')
    os.makedirs('/content/drive/MyDrive/logit-thermo' + '/A2_explore', exist_ok=True)
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo' + '/A2_explore/tm_salt_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 偽薬で T_m は下がるかで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：偽薬の T_m を A2_explore/tm_salt_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
